In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from datetime import datetime, timedelta

# =====================================================
# PARAMETERS
# =====================================================

gold_table = "payment_app.gold.fact_reconciliation"

# =====================================================
# READ SILVER TABLES
# =====================================================
bou = (
    spark.table("payment_app.silver.bou_transactions")
    .filter(F.col("batch_id") == v_batch_id)
)

cou = (
    spark.table("payment_app.silver.cou_transactions")
    .filter(F.col("batch_id") == v_batch_id)
)

# today's credit files
settlement = (
    spark.table("payment_app.silver.settlement")
    .filter(F.col("batch_id") == v_batch_id)
)

cbs = (
    spark.table("payment_app.silver.cbs")
    .filter(F.col("batch_id") == v_batch_id)
)

dim_biller = (
    spark.table("payment_app.gold.dim_biller")
    .filter("is_current = 'Y'")
)

# =====================================================
# CURRENT DAY RECON
# =====================================================

# BOU -> Settlement
fact_bou = (
    bou.alias("t")
    .join(settlement.alias("s"), "txn_id", "left")
    .join(dim_biller.alias("b"), "biller_id", "left")
    .select(
        col("txn_id"),
        date_format(to_date(col("t.transaction_time")), "yyyyMMdd").cast("int").alias("date_key"),
        col("b.biller_key"),
        lit("BOU").alias("txn_source"),
        col("t.status").alias("txn_status"),
        col("t.txn_amount"),
        coalesce(col("s.txn_amount"), lit(0.0)).alias("credited_amount"),
        when(col("s.txn_id").isNotNull(), "Y").otherwise("N").alias("recon_flag"),
        lit(v_batch_id).alias("batch_id"),
        current_timestamp().alias("created_timestamp"),
        current_timestamp().alias("updated_timestamp")
    )
)

# COU -> CBS
fact_cou = (
    cou.alias("t")
    .join(cbs.alias("c"), "txn_id", "left")
    .join(dim_biller.alias("b"), "biller_id", "left")
    .select(
        col("txn_id"),
        date_format(to_date(col("t.transaction_time")), "yyyyMMdd").cast("int").alias("date_key"),
        col("b.biller_key"),
        lit("COU").alias("txn_source"),
        col("t.status").alias("txn_status"),
        col("t.txn_amount"),
        coalesce(col("c.credit_amount"), lit(0.0)).alias("credited_amount"),
        when(col("c.txn_id").isNotNull(), "Y").otherwise("N").alias("recon_flag"),
        lit(v_batch_id).alias("batch_id"),
        current_timestamp().alias("created_timestamp"),
        current_timestamp().alias("updated_timestamp")
    )
)

fact_today = fact_bou.unionByName(fact_cou)

# =====================================================
# PREVIOUS DAY LATE ARRIVING RECON
# =====================================================
if spark.catalog.tableExists(gold_table):

    prev_day = (
        datetime.strptime(v_batch_id, "%Y-%m-%d") - timedelta(days=1)
    ).strftime("%Y%m%d")

    pending_prev = (
        spark.table(gold_table)
        .filter(
            (col("date_key") == int(prev_day)) &
            (col("recon_flag") == "N")
        )
    )

    # previous day BOU pending -> today's settlement
    late_bou = (
        pending_prev.filter(col("txn_source") == "BOU")
        .alias("f")
        .join(settlement.alias("s"), "txn_id", "inner")
        .select(
            col("f.txn_id"),
            col("f.date_key"),
            col("f.biller_key"),
            col("f.txn_source"),
            col("f.txn_status"),
            col("f.txn_amount"),
            col("s.txn_amount").alias("credited_amount"),
            lit("Y").alias("recon_flag"),
            lit(v_batch_id).alias("batch_id"),
            col("f.created_timestamp"),
            current_timestamp().alias("updated_timestamp")
        )
    )

    # previous day COU pending -> today's cbs
    late_cou = (
        pending_prev.filter(col("txn_source") == "COU")
        .alias("f")
        .join(cbs.alias("c"), "txn_id", "inner")
        .select(
            col("f.txn_id"),
            col("f.date_key"),
            col("f.biller_key"),
            col("f.txn_source"),
            col("f.txn_status"),
            col("f.txn_amount"),
            col("c.credit_amount").alias("credited_amount"),
            lit("Y").alias("recon_flag"),
            lit(v_batch_id).alias("batch_id"),
            col("f.created_timestamp"),
            current_timestamp().alias("updated_timestamp")
        )
    )

    fact_final = fact_today.unionByName(late_bou).unionByName(late_cou)

else:
    fact_final = fact_today

fact_final.createOrReplaceTempView("vw_fact_final")

# =====================================================
# FIRST LOAD
# =====================================================
if not spark.catalog.tableExists(gold_table):

    fact_final.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

# =====================================================
# INCREMENTAL MERGE
# =====================================================
else:

    spark.sql(f"""
        MERGE INTO {gold_table} tgt
        USING vw_fact_final src
        ON tgt.txn_id = src.txn_id

        WHEN MATCHED
        AND src.batch_id >= tgt.batch_id
        THEN UPDATE SET
            tgt.date_key          = src.date_key,
            tgt.biller_key        = src.biller_key,
            tgt.txn_source        = src.txn_source,
            tgt.txn_status        = src.txn_status,
            tgt.txn_amount        = src.txn_amount,
            tgt.credited_amount   = src.credited_amount,
            tgt.recon_flag        = src.recon_flag,
            tgt.batch_id          = src.batch_id,
            tgt.updated_timestamp = current_timestamp()

        WHEN NOT MATCHED
        THEN INSERT (
            txn_id,
            date_key,
            biller_key,
            txn_source,
            txn_status,
            txn_amount,
            credited_amount,
            recon_flag,
            batch_id,
            created_timestamp,
            updated_timestamp
        )
        VALUES (
            src.txn_id,
            src.date_key,
            src.biller_key,
            src.txn_source,
            src.txn_status,
            src.txn_amount,
            src.credited_amount,
            src.recon_flag,
            src.batch_id,
            src.created_timestamp,
            src.updated_timestamp
        )
    """)